# Pipeline of execution: 

* Run `resample.py`
* Run `features_volumetric.py`
* Run `features_radiomic.py`
* Run `features_displacement.py`

In [2]:
# Leitura radiomicos e volumetricos

import pandas as pd

ab = "abordagem_teste"

vol_path = f"/mnt/study-data/pgirardi/graphs/csvs/{ab}/features_volumetric_teste.csv"

rad_path = f"/mnt/study-data/pgirardi/graphs/csvs/{ab}/features_radiomic_teste.csv"

vol_df = pd.read_csv(vol_path)
rad_df = pd.read_csv(rad_path)


In [3]:
# Normalizacao pelo ICV (mask_mm3 global)

# ICV = volume total da mascara do encefalo (linha __global__ em features_volumetric)
# Vamos anexar o ICV por ID_IMG e normalizar apenas as features dependentes de tamanho.

import numpy as np

# 1) ICV por ID_IMG
vol_df["ID_IMG"] = vol_df["ID_IMG"].astype(str).str.strip()

icv_df = (
    vol_df.loc[vol_df["roi"].astype(str) == "__global__", ["ID_IMG", "mask_mm3"]]
    .drop_duplicates(subset=["ID_IMG"], keep="last")
    .rename(columns={"mask_mm3": "ICV_mask_mm3"})
)

icv_df["ICV_mask_mm3"] = pd.to_numeric(icv_df["ICV_mask_mm3"], errors="coerce")

# 2) Merge com radiomicos
rad_df["ID_IMG"] = rad_df["ID_IMG"].astype(str).str.strip()
merged = rad_df.merge(icv_df, on="ID_IMG", how="left", validate="many_to_one")

missing_icv = int(merged["ICV_mask_mm3"].isna().sum())
if missing_icv:
    raise ValueError(
        f"ICV ausente para {missing_icv} linhas radiomicas. "
        "Verifique se todos os ID_IMG do radiomico existem no volumetrico (linha __global__)."
    )

# 3) Features a normalizar (dependentes de tamanho)
cols_to_norm = [
    # first-order
    "original_firstorder_Energy",
    "original_firstorder_TotalEnergy",
    # shape (absolutas)
    "original_shape_MeshVolume",
    "original_shape_VoxelVolume",
    "original_shape_SurfaceArea",
    "original_shape_LeastAxisLength",
    "original_shape_MajorAxisLength",
    "original_shape_MinorAxisLength",
    "original_shape_Maximum2DDiameterColumn",
    "original_shape_Maximum2DDiameterRow",
    "original_shape_Maximum2DDiameterSlice",
    "original_shape_Maximum3DDiameter",
]

# mantém só as que existem no arquivo (robustez)
cols_to_norm = [c for c in cols_to_norm if c in merged.columns]

# garante numérico
for c in cols_to_norm:
    merged[c] = pd.to_numeric(merged[c], errors="coerce")

# 4) Normaliza pelo ICV
# Observacao: algumas features (p.ex. area) ficam com unidade relativa a volume;
# aqui seguimos a definicao pedida: dividir pelo volume total do encefalo.
merged[cols_to_norm] = merged[cols_to_norm].div(merged["ICV_mask_mm3"], axis=0)

# 5) CSV final: sem colunas "brutas" não-normalizadas adicionais (mantemos só a versão normalizada)
# (as colunas normalizadas sobrescrevem as originais, então basta salvar o merged)
out_path = f"/mnt/study-data/pgirardi/graphs/csvs/{ab}/radiomics_merge.csv"
merged.to_csv(out_path, index=False)

out_path


'/mnt/study-data/pgirardi/graphs/csvs/abordagem_teste/radiomics_merge.csv'

In [4]:
# Leitura dados técnicos e inserção dos equipamentos na planilha de atributos radiomicos

import pandas as pd

tech_path = f"/mnt/study-data/pgirardi/graphs/csvs/adnimerged.csv"

radiomics_merged_path = f"/mnt/study-data/pgirardi/graphs/csvs/{ab}/radiomics_merge.csv"

tech_df = pd.read_csv(tech_path)
radiomics_merged_df = pd.read_csv(radiomics_merged_path)

In [5]:
# Inserção de variáveis técnicas (por ID_IMG)

import numpy as np
import pandas as pd

# A planilha "merge" será o radiomics_merged_df enriquecido com informações técnicas/clínicas.
# (Se você preferir outro nome, basta trocar a variável aqui.)
merge = radiomics_merged_df.copy()

# 1) Merge por ID_IMG trazendo as colunas necessárias
cols_needed = [
    "ID_IMG",
    "ID_PT",
    "SEX",
    "DIAG",
    "MRI_DATE",
    "FIELD_STRENGTH",
    "SLICE_THICKNESS",
    "MANUFACTURER",
    "MFG_MODEL",
]

tech_sub = tech_df.loc[:, cols_needed].copy()
tech_sub["ID_IMG"] = tech_sub["ID_IMG"].astype(str).str.strip()
tech_sub = tech_sub.drop_duplicates(subset=["ID_IMG"], keep="last")

merge["ID_IMG"] = merge["ID_IMG"].astype(str).str.strip()
merge = merge.merge(tech_sub, on="ID_IMG", how="left", validate="many_to_one")

# 2) Define batch (scanner) a partir das variáveis do equipamento
merge["batch"] = (
    merge[["MANUFACTURER", "MFG_MODEL", "FIELD_STRENGTH", "SLICE_THICKNESS"]]
    .astype(str)
    .agg("_".join, axis=1)
)

missing_any = int(merge[cols_needed[1:]].isna().any(axis=1).sum())
print(f"Linhas sem alguma info técnica/clínica necessária (após merge): {missing_any}")



Linhas sem alguma info técnica/clínica necessária (após merge): 0


In [6]:
# Deltas radiômicos i12/i13/i23 + tempos t12/t13/t23 (em meses) usando apenas a planilha de conjuntos (cj_df)

import numpy as np
import pandas as pd

cj_path = f"/mnt/study-data/pgirardi/graphs/cj_data_{ab}.txt"

cj_df = pd.read_csv(cj_path)

radiomics_src = merge

# Chaves para ROI
roi_keys = ["ID_IMG", "roi", "side", "label"]
missing_keys = [c for c in roi_keys if c not in radiomics_src.columns]
if missing_keys:
    raise ValueError(f"radiomics_src não contém colunas necessárias: {missing_keys}")

# Radiômicos: somente colunas original_*
feat_cols = [c for c in radiomics_src.columns if c.startswith("original_")]
if not feat_cols:
    raise ValueError("Nenhuma coluna radiômica encontrada com prefixo 'original_'.")

# Base radiômica: 1 linha por (ID_IMG, roi, side, label)
base = radiomics_src.loc[:, roi_keys + feat_cols].copy()
base["ID_IMG"] = base["ID_IMG"].astype(str).str.strip()
base = base.drop_duplicates(subset=roi_keys, keep="last")

# --- triplets i1/i2/i3 a partir do cj_df (ordenado por MRI_DATE) ---
req = {"ID_PT", "COMBINATION_NUMBER", "ID_IMG", "MRI_DATE"}
if not req.issubset(set(cj_df.columns)):
    raise ValueError(f"cj_df precisa conter as colunas: {sorted(req)}")

cj = cj_df.copy()
cj["ID_IMG"] = cj["ID_IMG"].astype(str).str.strip()
cj["MRI_DATE"] = pd.to_datetime(cj["MRI_DATE"], errors="coerce")

trip = (
    cj.dropna(subset=["MRI_DATE"])
    .sort_values(["ID_PT", "COMBINATION_NUMBER", "MRI_DATE", "ID_IMG"])
    .groupby(["ID_PT", "COMBINATION_NUMBER"], as_index=False)
    .agg(
        ID_IMG_list=("ID_IMG", lambda s: list(pd.unique(s))),
        MRI_DATE_list=("MRI_DATE", lambda s: list(pd.unique(s))),
    )
)

trip["n_imgs"] = trip["ID_IMG_list"].apply(len)
trip = trip.loc[trip["n_imgs"] >= 3].copy()

trip[["ID_IMG_i1", "ID_IMG_i2", "ID_IMG_i3"]] = pd.DataFrame(trip["ID_IMG_list"].str[:3].to_list(), index=trip.index)
trip[["MRI_DATE_i1", "MRI_DATE_i2", "MRI_DATE_i3"]] = pd.DataFrame(trip["MRI_DATE_list"].str[:3].to_list(), index=trip.index)

# tempos em meses
DAYS_PER_MONTH = 365.2425 / 12.0
trip["t12"] = ((trip["MRI_DATE_i2"] - trip["MRI_DATE_i1"]).dt.total_seconds() / 86400.0) / DAYS_PER_MONTH
trip["t13"] = ((trip["MRI_DATE_i3"] - trip["MRI_DATE_i1"]).dt.total_seconds() / 86400.0) / DAYS_PER_MONTH
trip["t23"] = ((trip["MRI_DATE_i3"] - trip["MRI_DATE_i2"]).dt.total_seconds() / 86400.0) / DAYS_PER_MONTH

trip = trip.drop(columns=["ID_IMG_list", "MRI_DATE_list", "n_imgs", "MRI_DATE_i1", "MRI_DATE_i2", "MRI_DATE_i3"])

# --- anexa radiômicos de i1/i2/i3 (inner => sem NaN) ---
# Começa por i1 para materializar ROIs e depois intersecta com i2 e i3.

i1 = base.rename(columns={c: f"{c}_i1" for c in feat_cols})
tri = trip.loc[:, ["ID_PT", "COMBINATION_NUMBER", "ID_IMG_i1", "ID_IMG_i2", "ID_IMG_i3", "t12", "t13", "t23"]].copy()
# uma imagem (i1) expande para muitas ROIs => 1 -> N
tri = tri.merge(i1.rename(columns={"ID_IMG": "ID_IMG_i1"}), on="ID_IMG_i1", how="inner", validate="one_to_many")

# Função para anexar features mantendo a mesma ROI

def attach_base(df, img_col, suffix):
    tmp = base.rename(columns={c: f"{c}{suffix}" for c in feat_cols}).rename(columns={"ID_IMG": img_col})
    return df.merge(tmp, on=[img_col, "roi", "side", "label"], how="inner", validate="many_to_one")

tri = attach_base(tri, "ID_IMG_i2", "_i2")
tri = attach_base(tri, "ID_IMG_i3", "_i3")

# --- covariáveis por ID_IMG (GROUP/SEX/DIAG/AGE) ---
# Insere GROUP/SEX/DIAG/AGE por ID_IMG e exige consistência dentro do triplet
need_cols = ["ID_IMG", "GROUP", "SEX", "DIAG", "AGE"]
missing = [c for c in need_cols if c not in cj_df.columns]
if missing:
    raise ValueError(f"cj_df precisa conter as colunas: {missing}")

cov_map = cj_df.loc[:, need_cols].copy()
cov_map["ID_IMG"] = cov_map["ID_IMG"].astype(str).str.strip()
cov_map = cov_map.drop_duplicates(subset=["ID_IMG"], keep="last")

for img_col, suf in [("ID_IMG_i1", "_i1"), ("ID_IMG_i2", "_i2"), ("ID_IMG_i3", "_i3")]:
    tri = tri.merge(
        cov_map.rename(columns={"ID_IMG": img_col, "GROUP": f"GROUP{suf}", "SEX": f"SEX{suf}", "DIAG": f"DIAG{suf}", "AGE": f"AGE{suf}"}),
        on=img_col,
        how="left",
        validate="many_to_one",
    )

# Mantém as covariáveis da i1 e checa inconsistências no triplet
tri["GROUP"] = tri["GROUP_i1"]
tri["SEX"] = tri["SEX_i1"]
tri["DIAG"] = tri["DIAG_i1"]
tri["AGE"] = tri["AGE_i1"]

tri["_cov_mismatch"] = (
    (tri["GROUP_i1"] != tri["GROUP_i2"]) | (tri["GROUP_i1"] != tri["GROUP_i3"]) |
    (tri["SEX_i1"] != tri["SEX_i2"]) | (tri["SEX_i1"] != tri["SEX_i3"]) |
    (tri["DIAG_i1"] != tri["DIAG_i2"]) | (tri["DIAG_i1"] != tri["DIAG_i3"])
)
# print(f"Linhas com covariáveis inconsistentes entre i1/i2/i3: {int(tri['_cov_mismatch'].sum())}")

tri = tri.drop(columns=[
    "GROUP_i1", "GROUP_i2", "GROUP_i3",
    "SEX_i1", "SEX_i2", "SEX_i3",
    "DIAG_i1", "DIAG_i2", "DIAG_i3",
    "AGE_i1", "AGE_i2", "AGE_i3",
    "_cov_mismatch",
])

# --- calcula deltas e gera formato igual ao deslocamento (LONG por pair, colunas = atributos) ---
X1 = tri[[f"{c}_i1" for c in feat_cols]].apply(pd.to_numeric, errors="coerce").to_numpy()
X2 = tri[[f"{c}_i2" for c in feat_cols]].apply(pd.to_numeric, errors="coerce").to_numpy()
X3 = tri[[f"{c}_i3" for c in feat_cols]].apply(pd.to_numeric, errors="coerce").to_numpy()

D12 = X2 - X1
D13 = X3 - X1
D23 = X3 - X2

# metadados base (uma linha por ROI-triplet)
# IMPORTANTE: não podemos dropar colunas como `ID_IMG_i1` (termina com _i1). Então removemos
# apenas as colunas de FEATURES anexadas.
feat_drop = [f"{c}_i1" for c in feat_cols] + [f"{c}_i2" for c in feat_cols] + [f"{c}_i3" for c in feat_cols]
meta = tri.drop(columns=[c for c in feat_drop if c in tri.columns]).reset_index(drop=True)

# Garante colunas necessárias para merge com o deslocamento
if "TRIPLET_IDX" not in meta.columns:
    meta["TRIPLET_IDX"] = 0

# cria 3 tabelas (pair = 12/13/23) com as MESMAS colunas de atributos
f12 = pd.concat([meta.assign(pair=12), pd.DataFrame(D12, columns=feat_cols)], axis=1)
f13 = pd.concat([meta.assign(pair=13), pd.DataFrame(D13, columns=feat_cols)], axis=1)
f23 = pd.concat([meta.assign(pair=23), pd.DataFrame(D23, columns=feat_cols)], axis=1)

out = pd.concat([f12, f13, f23], ignore_index=True)

# opcional: deixa pair como string (se você preferir igual ao deslocamento)
# out["pair"] = out["pair"].astype(str)

# Reordena colunas: chaves -> covariáveis/tempos -> atributos radiômicos
key_cols = [
    "ID_PT",
    "COMBINATION_NUMBER",
    "TRIPLET_IDX",
    "pair",
    "ID_IMG_i1",
    "ID_IMG_i2",
    "ID_IMG_i3",
    "roi",
    "side",
    "label",
]
extra_cols = ["t12", "t13", "t23", "GROUP", "SEX", "DIAG", "AGE"]
feat_out_cols = feat_cols

ordered = [c for c in key_cols if c in out.columns] + [c for c in extra_cols if c in out.columns] + [c for c in feat_out_cols if c in out.columns]
# mantém qualquer coluna inesperada no fim
ordered += [c for c in out.columns if c not in ordered]
out = out.loc[:, ordered]

out_delta_path = f"/mnt/study-data/pgirardi/graphs/csvs/{ab}/deltas_radiomics.csv"
out.to_csv(out_delta_path, index=False)

out_delta_path

'/mnt/study-data/pgirardi/graphs/csvs/abordagem_teste/deltas_radiomics.csv'

# Pipeline

* Run `merge_strain_into_features_displacement.py`

In [10]:
# Merge: deslocamento (with strain) + deltas radiômicos

import pandas as pd
import numpy as np

rad_delta_path = f"/mnt/study-data/pgirardi/graphs/csvs/{ab}/deltas_radiomics.csv"
disp_path = f"/mnt/study-data/pgirardi/graphs/csvs/{ab}/features_displacement_teste_with_strain.csv"

rad_delta_df = pd.read_csv(rad_delta_path)
disp_df2 = pd.read_csv(disp_path)

# chaves comuns esperadas
keys = [
    "ID_PT",
    "COMBINATION_NUMBER",
    "TRIPLET_IDX",
    "pair",
    "ID_IMG_i1",
    "ID_IMG_i2",
    "ID_IMG_i3",
    "roi",
    "side",
    "label",
]

missing_rad = [c for c in keys if c not in rad_delta_df.columns]
missing_disp = [c for c in keys if c not in disp_df2.columns]
if missing_rad:
    raise ValueError(f"deltas_radiomics.csv sem colunas-chave: {missing_rad}")
if missing_disp:
    raise ValueError(f"displacement_with_strain.csv sem colunas-chave: {missing_disp}")

# garante tipos alinhados
rad_delta_df["pair"] = rad_delta_df["pair"].astype(str)
disp_df2["pair"] = disp_df2["pair"].astype(str)

# merge 1:1 esperado
merged_all = disp_df2.merge(
    rad_delta_df,
    on=keys,
    how="inner",
    suffixes=("", "_rad"),
    validate="one_to_one",
)

print(f"Linhas displacement: {len(disp_df2)}")
print(f"Linhas radiomics deltas: {len(rad_delta_df)}")
print(f"Linhas após merge (inner): {len(merged_all)}")

# Anexa covariáveis/infos técnicas POR IMAGEM de referência (ID_IMG_ref)
# Cada linha do delta (pair) referencia uma imagem "alvo":
# - pair 12: i2 (follow-up)
# - pair 13: i3
# - pair 23: i3
merged_all["pair"] = merged_all["pair"].astype(str)
merged_all["ID_IMG_ref"] = np.select(
    [merged_all["pair"].eq("12"), merged_all["pair"].eq("13"), merged_all["pair"].eq("23")],
    [merged_all["ID_IMG_i2"], merged_all["ID_IMG_i3"], merged_all["ID_IMG_i3"]],
    default=merged_all["ID_IMG_i2"],
)
merged_all["ID_IMG_ref"] = merged_all["ID_IMG_ref"].astype(str).str.strip()

# Remove AGE/SEX/DIAG vindos do pipeline anterior (triplet/i1). O merge com adnimerged
# traz os valores por ID_IMG_ref; sem esse drop o pandas duplica colunas -> SEX_x / SEX_y.
_drop_prev = [c for c in ("AGE", "DIAG", "SEX") if c in merged_all.columns]
if _drop_prev:
    merged_all = merged_all.drop(columns=_drop_prev)

tech_df = pd.read_csv("/mnt/study-data/pgirardi/graphs/csvs/adnimerged.csv")
tech_cols = [
    "ID_IMG",
    "AGE",
    "DIAG",
    "SEX",
    "FIELD_STRENGTH",
    "SLICE_THICKNESS",
    "MANUFACTURER",
    "MFG_MODEL",
]
tech_sub = tech_df.loc[:, tech_cols].copy()
tech_sub["ID_IMG"] = tech_sub["ID_IMG"].astype(str).str.strip()
tech_sub = tech_sub.drop_duplicates(subset=["ID_IMG"], keep="last")

merged_all = merged_all.merge(
    tech_sub.rename(columns={"ID_IMG": "ID_IMG_ref"}),
    on="ID_IMG_ref",
    how="left",
    validate="many_to_one",
)

# Opcional: batch pronto para neuroCombat
merged_all["batch"] = (
    merged_all[["MANUFACTURER", "MFG_MODEL", "FIELD_STRENGTH", "SLICE_THICKNESS"]]
    .astype(str)
    .agg("_".join, axis=1)
)

# Reorganiza colunas no merged final: chaves -> covariáveis/tempos -> técnicas -> features
key_cols = keys
extra_cols = ["t12", "t13", "t23", "GROUP", "SEX", "DIAG", "AGE"]
technical_cols = ["ID_IMG_ref", "FIELD_STRENGTH", "SLICE_THICKNESS", "MANUFACTURER", "MFG_MODEL", "batch"]

# separa features (tudo que não é chave nem extra nem técnico)
feature_cols = [c for c in merged_all.columns if c not in set(key_cols + extra_cols + technical_cols)]

ordered = [c for c in key_cols if c in merged_all.columns]
ordered += [c for c in extra_cols if c in merged_all.columns]
ordered += [c for c in technical_cols if c in merged_all.columns]
ordered += feature_cols
merged_all = merged_all.loc[:, ordered]

out_merged_path = f"/mnt/study-data/pgirardi/graphs/csvs/{ab}/all_delta_features.csv"
merged_all.to_csv(out_merged_path, index=False)

out_merged_path

Linhas displacement: 5520
Linhas radiomics deltas: 5520
Linhas após merge (inner): 5520


/mnt/study-data/pgirardi/tmp/ipykernel_3050654/1379808788.py:56: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  merged_all["ID_IMG_ref"] = np.select(
/mnt/study-data/pgirardi/tmp/ipykernel_3050654/1379808788.py:92: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  merged_all["batch"] = (


'/mnt/study-data/pgirardi/graphs/csvs/abordagem_teste/all_delta_features.csv'

In [11]:
# Harmonização com NeuroCombat (ComBat multi-site) — etapa final deste notebook.
# Próximo passo (modelagem): z-score com sklearn StandardScaler dentro de Pipeline + CV por fold,
# usando o CSV harmonizado como entrada (evita vazamento entre treino/validação).

# Entrada: all_delta_features.csv (mesmo `ab` das células anteriores).
# Saída: colunas de metadados inalteradas + features ajustadas por scanner (`batch`).

import numpy as np
import pandas as pd

from neuroCombat import neuroCombat

in_path = f"/mnt/study-data/pgirardi/graphs/csvs/{ab}/all_delta_features.csv"
out_path = f"/mnt/study-data/pgirardi/graphs/csvs/{ab}/all_delta_features_neurocombat.csv"

key_cols = [
    "ID_PT",
    "COMBINATION_NUMBER",
    "TRIPLET_IDX",
    "pair",
    "ID_IMG_i1",
    "ID_IMG_i2",
    "ID_IMG_i3",
    "roi",
    "side",
    "label",
]
extra_cols = ["t12", "t13", "t23", "GROUP", "SEX", "DIAG", "AGE"]
technical_cols = ["ID_IMG_ref", "FIELD_STRENGTH", "SLICE_THICKNESS", "MANUFACTURER", "MFG_MODEL", "batch"]

meta_cols = [c for c in key_cols + extra_cols + technical_cols]

df = pd.read_csv(in_path)
missing_meta = [c for c in meta_cols if c not in df.columns]
if missing_meta:
    raise ValueError(f"CSV sem colunas esperadas: {missing_meta}")

feature_cols = [c for c in df.columns if c not in meta_cols]
if not feature_cols:
    raise ValueError("Nenhuma coluna numérica (feature) encontrada além dos metadados.")

X = df[feature_cols].apply(pd.to_numeric, errors="coerce")

# Covariáveis biológicas preservadas no modelo além do batch (NeuroCombat usa design matrix)
covar_cols = ["batch", "AGE", "SEX", "DIAG"]
for c in covar_cols:
    if c not in df.columns:
        raise ValueError(f"Coluna necessária ausente: {c}")

covars = df[covar_cols].copy()
covars["AGE"] = pd.to_numeric(covars["AGE"], errors="coerce")

mask = covars["batch"].notna().astype(bool)
mask &= X.notna().all(axis=1)
mask &= covars["AGE"].notna()
mask &= covars["SEX"].notna()
mask &= covars["DIAG"].notna()

n_drop = int((~mask).sum())
if n_drop:
    print(f"Linhas excluídas (batch/features/covars incompletos): {n_drop}")

df_fit = df.loc[mask].reset_index(drop=True)
covars_fit = covars.loc[mask].reset_index(drop=True)
X_fit = X.loc[mask].reset_index(drop=True)

# neuroCombat espera dat com shape (n_features, n_amostras)
dat = X_fit.to_numpy(dtype=float).T

n_batch = covars_fit["batch"].nunique()
print(f"Amostras na harmonização: {dat.shape[1]} | Features: {dat.shape[0]} | Níveis de batch: {n_batch}")
if n_batch < 2:
    raise ValueError(
        "NeuroCombat requer pelo menos 2 batches distintos. Verifique a coluna batch."
    )

res = neuroCombat(
    dat=dat,
    covars=covars_fit,
    batch_col="batch",
    categorical_cols=["SEX", "DIAG"],
    eb=True,
    parametric=True,
    mean_only=False,
)

harm = res["data"].T  # volta para (n_amostras, n_features)
harm_df = pd.DataFrame(harm, columns=feature_cols, index=df_fit.index)

out = pd.concat([df_fit[meta_cols], harm_df], axis=1)
# mantém a mesma ordem de colunas do arquivo de entrada (features harmonizadas no lugar das originais)
out = out.loc[:, df.columns]

out.to_csv(out_path, index=False)
print(f"Salvo: {out_path}")

out_path

Amostras na harmonização: 5520 | Features: 194 | Níveis de batch: 11
[neuroCombat] Creating design matrix
[neuroCombat] Standardizing data across features
[neuroCombat] Fitting L/S model and finding priors
[neuroCombat] Finding parametric adjustments
[neuroCombat] Final adjustment of data
Salvo: /mnt/study-data/pgirardi/graphs/csvs/abordagem_teste/all_delta_features_neurocombat.csv


'/mnt/study-data/pgirardi/graphs/csvs/abordagem_teste/all_delta_features_neurocombat.csv'